In [38]:
import numpy as np

<h2>Introduction</h2>

Structured arrays are ndarrays whose datatype is a composition of simpler datatypes organized as a sequence of named fields. For example,

In [39]:
x = np.array([('Rex', 9, 81.0),('Fido', 3, 27.0)],
            dtype=[('name','U10'),('age','i4'),('weight','f4')])
x

array([('Rex', 9, 81.), ('Fido', 3, 27.)],
      dtype=[('name', '<U10'), ('age', '<i4'), ('weight', '<f4')])

In [40]:
x[1]

np.void(('Fido', 3, 27.0), dtype=[('name', '<U10'), ('age', '<i4'), ('weight', '<f4')])

In [41]:
x['age']

array([9, 3], dtype=int32)

In [42]:
x['age'] =5
x

array([('Rex', 5, 81.), ('Fido', 5, 27.)],
      dtype=[('name', '<U10'), ('age', '<i4'), ('weight', '<f4')])

<h2>Structured datatypes</h2>
<h3>Structured datatype creation</h3>

In [43]:
# a list of tuples, one tuple per field
np.dtype([('x','f4'),('y', np.float32), ('z', 'f4', (2,2))])

dtype([('x', '<f4'), ('y', '<f4'), ('z', '<f4', (2, 2))])

In [44]:
#If fieldname is the empty string '', the field will be given a default name of the form f#
np.dtype([('x','f4'),('','i4'),('z','i8')])

dtype([('x', '<f4'), ('f1', '<i4'), ('z', '<i8')])

In [45]:
#A string of comma-separated dtype specifications
np.dtype('i8, f4, S3')

dtype([('f0', '<i8'), ('f1', '<f4'), ('f2', 'S3')])

In [46]:
np.dtype('3int8, float32, (2, 3)float64')

dtype([('f0', 'i1', (3,)), ('f1', '<f4'), ('f2', '<f8', (2, 3))])

In [47]:
# A dictionary of field parameter arrays
np.dtype({'names':['col1','col2'], 'formats':['i4', 'f4']})

dtype([('col1', '<i4'), ('col2', '<f4')])

In [48]:
np.dtype({'names':['col1','col2'],
         'formats':['i4','f4'],
         'offsets': [0, 4],
         'itemsize':12})

dtype({'names': ['col1', 'col2'], 'formats': ['<i4', '<f4'], 'offsets': [0, 4], 'itemsize': 12})

In [49]:
# A dictionary of field names
np.dtype({'col1':('i1',0), 'col2':('f4',1)})

dtype([('col1', 'i1'), ('col2', '<f4')])

This form was discouraged because Python dictionaries did not preserve order in Python versions before Python 3.6. Field Titles may be specified by using a 3-tuple, see below.

<h3>Manipulating and displaying structured datatypes</h3>

In [50]:
d = np.dtype([('x','i8'),('y','f4')])
d.names

('x', 'y')

In [51]:
d['x']

dtype('int64')

In [52]:
d.fields

mappingproxy({'x': (dtype('int64'), 0), 'y': (dtype('float32'), 8)})

<h3>Automatic byte offsets and alignment</h3>

In [53]:
def print_offsets(d):
    print("offsets:",[d.fields[name][1] for name in d.names])
    print("itemsize:", d.itemsize)

print_offsets(np.dtype('u1, u1, i4, u1, i8, u2'))

offsets: [0, 1, 2, 6, 7, 15]
itemsize: 17


In [54]:
print_offsets(np.dtype('u1, u1, i4, u1, i8, u2', align=True))

offsets: [0, 1, 4, 8, 16, 24]
itemsize: 32


<h3>Field titles</h3>

In [55]:
np.dtype([(('my_title', 'name'),'f4')])

dtype([(('my_title', 'name'), '<f4')])

using the second (discouraged) dictionary-based specification

In [56]:
np.dtype({'name':('i4', 0, 'my_title')})

dtype([(('my_title', 'name'), '<i4')])

it is recommended to iterate through the fields of a dtype using the names attribute of the dtype, which will not list titles, as in:

In [57]:
for name in d.names:
    print(d.fields[name][:2])

(dtype('int64'), 0)
(dtype('float32'), 8)


<h3>Union types</h3>

In [58]:
# https://www.geeksforgeeks.org/python/find-the-union-of-two-numpy-arrays/
a = np.array([10, 20, 30, 40])
b = np.array([20, 40, 60, 80])

res = np.union1d(a,b)
res

array([10, 20, 30, 40, 60, 80])

<h2>Indexing and assignment to structured arrays</h2>
<h3>Assigning data to a structured array</h3>
<h4>Assignment from Python Native Types (Tuples)</h4>

In [59]:
x= np.array([(1, 2, 3),(4, 5, 6)],dtype='i8, f4, f8')

In [60]:
x[1] = (7, 8, 9)

In [61]:
x

array([(1, 2., 3.), (7, 8., 9.)],
      dtype=[('f0', '<i8'), ('f1', '<f4'), ('f2', '<f8')])

<h4>Assignment from Scalars</h4>
A scalar assigned to a structured element will be assigned to all fields. This happens when a scalar is assigned to a structured array, or when an unstructured array is assigned to a structured array:

In [62]:
x = np.zeros(2, dtype='i8, f4, ?, S1')
x[:] =3
x

array([(3, 3.,  True, b'3'), (3, 3.,  True, b'3')],
      dtype=[('f0', '<i8'), ('f1', '<f4'), ('f2', '?'), ('f3', 'S1')])

In [63]:
x[:] = np.arange(2)
x

array([(0, 0., False, b'0'), (1, 1.,  True, b'1')],
      dtype=[('f0', '<i8'), ('f1', '<f4'), ('f2', '?'), ('f3', 'S1')])

Structured arrays can also be assigned to unstructured arrays, but only if the structured datatype has just a single field:

In [64]:
twofield = np.zeros(2, dtype =[('A','i4'),('B', 'i4')])
onefield = np.zeros(2, dtype =[('A','i4')])
nostruct = np.zeros(2, dtype = 'i4')
nostruct[:] = twofield

TypeError: Cannot cast array data from dtype([('A', '<i4'), ('B', '<i4')]) to dtype('int32') according to the rule 'unsafe'

<h3>Assignment from other Structured Arrays</h3>
Assignment between two structured arrays occurs as if the source elements had been converted to tuples and then assigned to the destination elements.

In [65]:
a = np.zeros(3, dtype=[('a','i8'),('b','f4'),('c','S3')])
b = np.ones(3, dtype=[('x','f4'),('y', 'S3'),('z', 'O')])
b[:]=a
b

array([(0., b'0.0', b''), (0., b'0.0', b''), (0., b'0.0', b'')],
      dtype=[('x', '<f4'), ('y', 'S3'), ('z', 'O')])

<h4>Assignment involving subarrays</h4>
When assigning to fields which are subarrays, the assigned value will first be broadcast to the shape of the subarray.

In [66]:
#sOURCE : YouTube
array1 = np.array([[1,2,3,4]])
array2 = np.array([[1],[2],[3],[4]])
array1.shape


(1, 4)

In [67]:
array2.shape

(4, 1)

In [68]:
print(array1 * array2)

[[ 1  2  3  4]
 [ 2  4  6  8]
 [ 3  6  9 12]
 [ 4  8 12 16]]


<h3>Indexing structured arrays</h3>

<h4>Accessing Individual Fields</h4>

In [69]:
x= np.array([(1, 2),(3, 4)], dtype=[('foo','i8'),('bar','f4')])

In [70]:
x['foo']

array([1, 3])

In [71]:
x['foo'] = 10

In [72]:
x

array([(10, 2.), (10, 4.)], dtype=[('foo', '<i8'), ('bar', '<f4')])

The resulting array is a view into the original array. It shares the same memory locations and writing to the view will modify the original array.

In [73]:
y = x['bar']
y[:]= 11
x

array([(10, 11.), (10, 11.)], dtype=[('foo', '<i8'), ('bar', '<f4')])

In [74]:
#strides > a tuple where each value represents the number of bytes to move in memory to access the next element along each dimension

y.dtype, y.shape, y.strides

(dtype('float32'), (2,), (12,))

If the accessed field is a subarray, the dimensions of the subarray are appended to the shape of the result:

In [75]:
x = np.zeros((2,2), dtype =[('a', np.int32),('b',np.float64, (3,3))])

In [76]:
x['a'].shape

(2, 2)

In [77]:
x['b'].shape

(2, 2, 3, 3)

<h4>Accessing Multiple Fields</h4>
One can index and assign to a structured array with a multi-field index, where the index is a list of field names.

In [78]:
a= np.zeros(3, dtype=[('a', 'i4'),('b','i4'),('c','f4')])
a[['a','c']]

array([(0, 0.), (0, 0.), (0, 0.)],
      dtype={'names': ['a', 'c'], 'formats': ['<i4', '<f4'], 'offsets': [0, 8], 'itemsize': 12})

Assignment to an array with a multi-field index modifies the original array:

In [79]:
a[['a', 'c']] = (2, 3)
a

array([(2, 0, 3.), (2, 0, 3.), (2, 0, 3.)],
      dtype=[('a', '<i4'), ('b', '<i4'), ('c', '<f4')])

In [80]:
a[['a','c']] = a[['c', 'a']]

<h4>Indexing with an Integer to get a Structured Scalar</h4>
Indexing a single element of a structured array (with an integer index) returns a structured scalar:

In [81]:
x=np.array([(1, 2., 3.)], dtype='i, f, f')
scaler = x[0]
scaler

np.void((1, 2.0, 3.0), dtype=[('f0', '<i4'), ('f1', '<f4'), ('f2', '<f4')])

In [82]:
type(scaler)

numpy.void

Unlike other numpy scalars, structured scalars are mutable and act like views into the original array

In [83]:
x = np.array([(1, 2),(3, 4)], dtype=[('foo', 'i8'),('bar', 'f4')])
s = x[0]
s['bar'] = 100
x

array([(1, 100.), (3,   4.)], dtype=[('foo', '<i8'), ('bar', '<f4')])

Similarly to tuples, structured scalars can also be indexed with an integer:

In [84]:
scaler = np.array([(1, 2., 3.)], dtype= 'i, f, f')[0]
scaler[0]

np.int32(1)

In [85]:
scaler[1]= 4

In [86]:
scaler

np.void((1, 4.0, 3.0), dtype=[('f0', '<i4'), ('f1', '<f4'), ('f2', '<f4')])

Structured scalars may be converted to a tuple by calling numpy.ndarray.item:

In [87]:
scaler.item(), type(scaler.item())

((1, 4.0, 3.0), tuple)